# Cenário ZT502PO | Universo PO completo

| Item | Valor |
|---|---|
| Objeto | ``dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido``; tabela DEV, somente leitura |
| DESCRIBE | `DESCRIBE_ZT502PO.csv`: 12 colunas; tipos STRING, INT e DATE; clustering: não informado |
| Filtro | `cod_componente_versao = 'PO'` (tipo de documento, **não centro nem empresa**) |
| Volume esperado | **29.300 linhas**, medido no HTML executado; recontar no momento da execução |
| Chave candidata | `num_documento + cod_componente_versao`; 29.300 chaves e nenhuma repetição no HTML |
| Lacunas medidas | `cod_motivo`, `nm_responsavel_pela_modificacao_objeto` e `qt_contagem` 100% nulos, documentados no DESCRIBE |
| SAP | Confirmar filtros do relatório ZT502PO antes de alegar equivalência; a curated não aplica filtros de negócio da tela |

### Alertas
| Severidade | Observação |
|---|---|
| 🔴 ALTA | PO completo no Datalake não equivale necessariamente ao relatório SAP: o comentário do objeto afirma que filtros ABAP, como documento não eliminado, não são aplicados na curated. |
| 🔴 ALTA | `nm_usuario` é dado sensível; amostras e extração usam somente hash. |
| 🟡 MÉDIA | Campos 100% nulos estão documentados como lacuna; não tratá-los como erro novo. |
| 🟡 MÉDIA | `dh_modificacao` é texto HHMMSS; `dateingest` é data de carga, não data de negócio. |

**Como usar:** executar Run All no Databricks. Cada consulta de dados tem o filtro PO embutido, não escreve dados e não depende de widgets. A seção 21 extrai o universo PO; uma extração equivalente à tela SAP depende de conhecer os filtros da tela e não foi inventada. O nome usa `PO` no lugar de `CentroNNNN` porque não há coluna de centro no schema.

### Índice
0 Escopo; 1 Metadados; 2 Volume; 3 Recorte PO; 4 Chave; 5 Duplicidade; 6 Preenchimento; 7 Cardinalidade; 8 Domínio; 9 Indicadores; 10 Numéricos; 11 Totais; 12 Datas; 13 Códigos; 14 Checksum; 15 Amostra; 16 Distribuição; 17 Freshness; 18 Regras; 19 Amostra ampliada; 20 Resumo; 21 Extração completa.

## 0. Escopo

O filtro PO é de tipo de documento, não substitui um centro SAP.

In [0]:
SELECT 'ZT502PO' AS transacao, 'PO' AS tipo_documento, 'sem cod_centro e cod_empresa' AS limitacao;

## 1. Metadados

Confere o comentário do objeto e os campos efetivamente publicados.

In [0]:
DESCRIBE EXTENDED `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`;

In [0]:
DESCRIBE DETAIL `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`;

In [0]:
DESCRIBE HISTORY `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido`;

## 2. Volumetria

Confere o total PO e os documentos distintos no snapshot atual.

In [0]:
SELECT COUNT(*) AS linhas_atuais,29300 AS linhas_esperadas_exploratoria,COUNT(DISTINCT `num_documento`) AS documentos_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO';

## 3. Recorte PO

Confirma que somente o tipo PO foi incluído.

In [0]:
SELECT `cod_componente_versao`,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' GROUP BY `cod_componente_versao`;

## 4. Granularidade

Compara chave simples e chave composta sem colapsar NULL.

In [0]:
WITH a AS (SELECT `num_documento` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' GROUP BY `num_documento`),b AS (SELECT `num_documento`,`cod_componente_versao` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' GROUP BY `num_documento`,`cod_componente_versao`) SELECT (SELECT COUNT(*) FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO') AS linhas,(SELECT COUNT(*) FROM a) AS documentos,(SELECT COUNT(*) FROM b) AS chaves_compostas;

## 5. Duplicatas idênticas e versões

Hash interno da linha inteira separa duplicatas puras de versões sem expor usuários.

In [0]:
WITH h AS (SELECT t.`num_documento`,t.`cod_componente_versao`,MD5(TO_JSON(STRUCT(t.`num_documento`, t.`cod_componente_versao`, t.`cod_liberacao`, t.`cod_motivo`, t.`nm_responsavel_pela_modificacao_objeto`, t.`nm_usuario`, t.`dt_ultima_modificacao`, t.`dh_modificacao`, t.`qt_contagem`, t.`dateingest`, t.`yearingest`, t.`monthingest`), MAP('ignoreNullFields','false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` t WHERE `cod_componente_versao` = 'PO'),d AS (SELECT `num_documento`,`cod_componente_versao`,COUNT(*) AS linhas,COUNT(DISTINCT hash_linha) AS versoes FROM h GROUP BY `num_documento`,`cod_componente_versao` HAVING COUNT(*)>1) SELECT COUNT(*) AS chaves_repetidas,COALESCE(SUM(linhas-versoes),0) AS copias_identicas,COUNT_IF(versoes>1) AS chaves_com_versoes FROM d;

## 6. Preenchimento de todas as colunas

Perfil de nulos, vazios e zeros para todas as colunas, sem valores literais de pessoas.

In [0]:
SELECT p.*,total-nulos-vazios AS preenchidos,CASE WHEN total=0 THEN 'SEM LINHAS' WHEN nulos=total THEN '100% NULO: CONFERIR COMENTARIO' WHEN nulos+vazios=total THEN '100% SEM VALOR' ELSE 'COM DADOS' END AS veredito FROM (SELECT 'num_documento' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`num_documento` IS NULL) AS nulos,COUNT_IF(`num_documento` IS NOT NULL AND LOWER(TRIM(`num_documento`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`num_documento`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' UNION ALL SELECT 'cod_componente_versao' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`cod_componente_versao` IS NULL) AS nulos,COUNT_IF(`cod_componente_versao` IS NOT NULL AND LOWER(TRIM(`cod_componente_versao`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`cod_componente_versao`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' UNION ALL SELECT 'cod_liberacao' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`cod_liberacao` IS NULL) AS nulos,COUNT_IF(`cod_liberacao` IS NOT NULL AND LOWER(TRIM(`cod_liberacao`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`cod_liberacao`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' UNION ALL SELECT 'cod_motivo' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`cod_motivo` IS NULL) AS nulos,COUNT_IF(`cod_motivo` IS NOT NULL AND LOWER(TRIM(`cod_motivo`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`cod_motivo`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' UNION ALL SELECT 'nm_responsavel_pela_modificacao_objeto' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`nm_responsavel_pela_modificacao_objeto` IS NULL) AS nulos,COUNT_IF(`nm_responsavel_pela_modificacao_objeto` IS NOT NULL AND LOWER(TRIM(`nm_responsavel_pela_modificacao_objeto`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`nm_responsavel_pela_modificacao_objeto`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' UNION ALL SELECT 'nm_usuario' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`nm_usuario` IS NULL) AS nulos,COUNT_IF(`nm_usuario` IS NOT NULL AND LOWER(TRIM(`nm_usuario`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`nm_usuario`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' UNION ALL SELECT 'dt_ultima_modificacao' AS coluna,'date' AS tipo,COUNT(*) AS total,COUNT_IF(`dt_ultima_modificacao` IS NULL) AS nulos,CAST(0 AS BIGINT) AS vazios,CAST(0 AS BIGINT) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' UNION ALL SELECT 'dh_modificacao' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`dh_modificacao` IS NULL) AS nulos,COUNT_IF(`dh_modificacao` IS NOT NULL AND LOWER(TRIM(`dh_modificacao`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`dh_modificacao`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' UNION ALL SELECT 'qt_contagem' AS coluna,'int' AS tipo,COUNT(*) AS total,COUNT_IF(`qt_contagem` IS NULL) AS nulos,CAST(0 AS BIGINT) AS vazios,COUNT_IF(`qt_contagem`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' UNION ALL SELECT 'dateingest' AS coluna,'date' AS tipo,COUNT(*) AS total,COUNT_IF(`dateingest` IS NULL) AS nulos,CAST(0 AS BIGINT) AS vazios,CAST(0 AS BIGINT) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' UNION ALL SELECT 'yearingest' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`yearingest` IS NULL) AS nulos,COUNT_IF(`yearingest` IS NOT NULL AND LOWER(TRIM(`yearingest`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`yearingest`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' UNION ALL SELECT 'monthingest' AS coluna,'string' AS tipo,COUNT(*) AS total,COUNT_IF(`monthingest` IS NULL) AS nulos,COUNT_IF(`monthingest` IS NOT NULL AND LOWER(TRIM(`monthingest`)) IN ('','null','nan','none','#','-')) AS vazios,COUNT_IF(TRIM(`monthingest`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO') p ORDER BY coluna;

## 6.1. Lacunas documentadas

Confirma no recorte os três campos 100% nulos na exploratória.

In [0]:
SELECT * FROM (SELECT 'cod_motivo' AS coluna,COUNT(*) AS total,COUNT_IF(`cod_motivo` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' UNION ALL SELECT 'nm_responsavel_pela_modificacao_objeto' AS coluna,COUNT(*) AS total,COUNT_IF(`nm_responsavel_pela_modificacao_objeto` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' UNION ALL SELECT 'qt_contagem' AS coluna,COUNT(*) AS total,COUNT_IF(`qt_contagem` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO') x ORDER BY coluna;

## 7. Cardinalidade

Confere identificadores e códigos de liberação sem dados pessoais.

In [0]:
SELECT COUNT(DISTINCT `num_documento`) AS documentos,COUNT(DISTINCT `cod_componente_versao`) AS tipos,COUNT(DISTINCT `cod_liberacao`) AS codigos_liberacao FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO';

## 8. Domínio de liberação

Contagem por código de liberação; código não é necessariamente status de aprovação.

In [0]:
SELECT `cod_liberacao`,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' GROUP BY `cod_liberacao` ORDER BY linhas DESC LIMIT 20;

## 9. Indicadores

Mostra presença de liberação e usuário apenas por contagem.

In [0]:
SELECT COUNT(*) AS linhas,COUNT_IF(`cod_liberacao` IS NULL OR TRIM(`cod_liberacao`)='') AS sem_codigo_liberacao,COUNT_IF(`nm_usuario` IS NULL OR TRIM(`nm_usuario`)='') AS sem_usuario FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO';

## 10. Perfil numérico

`qt_contagem` é a única medida INT no schema e estava nula na exploratória.

In [0]:
SELECT MIN(`qt_contagem`) AS minimo,MAX(`qt_contagem`) AS maximo,AVG(`qt_contagem`) AS media,COUNT_IF(`qt_contagem`=0) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO';

## 11. Totais

Não interpretar a soma como valor financeiro.

In [0]:
SELECT COUNT(*) AS linhas,SUM(`qt_contagem`) AS soma_contagem FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO';

## 12. Datas e hora

`dt_ultima_modificacao` é DATE; `dh_modificacao` é STRING HHMMSS e não deve ser comparada como data.

In [0]:
SELECT MIN(`dt_ultima_modificacao`) AS primeira_modificacao,MAX(`dt_ultima_modificacao`) AS ultima_modificacao,COUNT_IF(`dh_modificacao` IS NOT NULL AND NOT (`dh_modificacao` RLIKE '^[0-9]{6}$')) AS horas_formato_invalido,COUNT_IF(`dh_modificacao` IS NULL) AS horas_nulas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO';

## 13. Códigos e zeros à esquerda

Mede códigos de pedido com zeros e colisões potenciais após normalização.

In [0]:
SELECT COUNT_IF(TRIM(`num_documento`) RLIKE '^0+[0-9]+$') AS pedidos_com_zeros,COUNT(DISTINCT CASE WHEN TRIM(`num_documento`) RLIKE '^[0-9]+$' THEN TRIM(`num_documento`) END) AS codigos_numericos,COUNT(DISTINCT CASE WHEN TRIM(`num_documento`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_documento`),'^0+(?=[0-9])','') END) AS codigos_normalizados FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO';

## 14. Checksum

Conta linhas 100% idênticas por hash interno.

In [0]:
WITH h AS (SELECT MD5(TO_JSON(STRUCT(t.`num_documento`, t.`cod_componente_versao`, t.`cod_liberacao`, t.`cod_motivo`, t.`nm_responsavel_pela_modificacao_objeto`, t.`nm_usuario`, t.`dt_ultima_modificacao`, t.`dh_modificacao`, t.`qt_contagem`, t.`dateingest`, t.`yearingest`, t.`monthingest`), MAP('ignoreNullFields','false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` t WHERE `cod_componente_versao` = 'PO'),d AS (SELECT hash_linha,COUNT(*) AS linhas FROM h GROUP BY hash_linha) SELECT COALESCE(SUM(linhas),0) AS linhas,COUNT(*) AS hashes_distintos,COALESCE(SUM(linhas-1),0) AS copias_identicas FROM d;

## 15. Amostra

20 linhas, com dados pessoais mascarados.

In [0]:
SELECT `num_documento`, `cod_componente_versao`, `cod_liberacao`, `cod_motivo`, `dt_ultima_modificacao`, `dh_modificacao`, `qt_contagem`, `dateingest`, `yearingest`, `monthingest`, SUBSTRING(MD5(CAST(`nm_responsavel_pela_modificacao_objeto` AS STRING)),1,8) AS `nm_responsavel_pela_modificacao_objeto_hash`, SUBSTRING(MD5(CAST(`nm_usuario` AS STRING)),1,8) AS `nm_usuario_hash` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' ORDER BY `num_documento` LIMIT 20;

## 16. Distribuição interna

Distribui por liberação e data de modificação sem expor usuário.

In [0]:
SELECT `cod_liberacao`,`dt_ultima_modificacao`,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' GROUP BY `cod_liberacao`,`dt_ultima_modificacao` ORDER BY linhas DESC LIMIT 100;

## 17. Freshness

Data de ingestão é data de carga, não data do pedido.

In [0]:
SELECT `dateingest`,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' GROUP BY `dateingest` ORDER BY `dateingest` DESC;

## 18. Regras específicas

Verifica chaves nulas, código de liberação e hora sem presumir filtros de negócio ABAP.

In [0]:
SELECT COUNT(*) AS linhas,COUNT_IF(`num_documento` IS NULL OR TRIM(`num_documento`)='') AS pedido_ausente,COUNT_IF(`cod_liberacao` IS NULL OR TRIM(`cod_liberacao`)='') AS liberacao_ausente,COUNT_IF(`dh_modificacao` IS NOT NULL AND NOT (`dh_modificacao` RLIKE '^[0-9]{6}$')) AS hora_formato_invalido FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO';

## 19. Amostra ampliada

Até 500 linhas, sem usuário literal.

In [0]:
SELECT `num_documento`, `cod_componente_versao`, `cod_liberacao`, `cod_motivo`, `dt_ultima_modificacao`, `dh_modificacao`, `qt_contagem`, `dateingest`, `yearingest`, `monthingest`, SUBSTRING(MD5(CAST(`nm_responsavel_pela_modificacao_objeto` AS STRING)),1,8) AS `nm_responsavel_pela_modificacao_objeto_hash`, SUBSTRING(MD5(CAST(`nm_usuario` AS STRING)),1,8) AS `nm_usuario_hash` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' ORDER BY `num_documento` LIMIT 500;

## 20. Resumo do canário

Concilia volume e chave do universo PO; não afirma equivalência com a tela SAP.

In [0]:
WITH d AS (SELECT `num_documento`,`cod_componente_versao`,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' GROUP BY `num_documento`,`cod_componente_versao`) SELECT 29300 AS linhas_esperadas_exploratoria,(SELECT COUNT(*) FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO') AS linhas_atuais,(SELECT COUNT(*) FROM d) AS chaves_distintas,(SELECT COUNT(*) FROM d WHERE linhas>1) AS chaves_repetidas;

## 21. EXTRAÇÃO COMPLETA PO

Sem LIMIT, todos os pedidos PO da base. Não é extração equivalente à tela SAP até que filtros de negócio sejam confirmados.

In [0]:
SELECT CASE WHEN TRIM(`num_documento`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_documento`),'^0+(?=[0-9])','') ELSE TRIM(`num_documento`) END AS `num_documento_normalizado`,`num_documento`, `cod_componente_versao`, `cod_liberacao`, `cod_motivo`, `dt_ultima_modificacao`, `dh_modificacao`, `qt_contagem`, `dateingest`, `yearingest`, `monthingest`, SUBSTRING(MD5(CAST(`nm_responsavel_pela_modificacao_objeto` AS STRING)),1,8) AS `nm_responsavel_pela_modificacao_objeto_hash`, SUBSTRING(MD5(CAST(`nm_usuario` AS STRING)),1,8) AS `nm_usuario_hash` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zt502po_estrategia_liberacao_pedido` WHERE `cod_componente_versao` = 'PO' ORDER BY `num_documento`,`cod_componente_versao`;

## Checklist SAP

- [ ] Rodar o notebook e registrar instante e data da carga.
- [ ] Comparar volume atual com 29.300 e conferir chave `num_documento + cod_componente_versao`.
- [ ] Confirmar os filtros do relatório ABAP, incluindo eventual exclusão de documentos eliminados; a tabela PO não os aplica automaticamente.
- [ ] Somente depois de conhecer os filtros SAP, construir a extração equivalente; não presumir que 29.300 é o volume da tela.
- [ ] Conferir os três campos documentados como nulos e não exportar identificadores pessoais literais.